In [ ]:
using HDF5
using PyPlot
using ProgressBars
using JLD2

In [ ]:
function load_imgs(filenames, pathdir)
    imgs_atoms, imgs_bkg, imgs_dark = [], [], []

    for filename in filenames
        filename = pathdir*filename
        h5open(filename, "r") do file
            img_atoms = convert(Matrix{Int}, read(file["images/Vertical_Axis_Camera/in_situ_absorption/atoms"]))
            img_bkg = convert(Matrix{Int}, read(file["images/Vertical_Axis_Camera/in_situ_absorption/background"]))
            img_dark = convert(Matrix{Int}, read(file["images/Vertical_Axis_Camera/in_situ_absorption/dark"]))
            push!(imgs_atoms, img_atoms), push!(imgs_bkg, img_bkg), push!(imgs_dark, img_dark)
        end
    end
    return imgs_atoms, imgs_bkg, imgs_dark
end

function crop_imgs(imgs, x_crop, y_crop)
    imgs_crop = []
    for i = 1:length(imgs)
        img_crop = imgs[i][y_crop, x_crop]
        push!(imgs_crop, img_crop)
    end
    return imgs_crop
end

function compute_OD(imgs_atoms, imgs_bkg, imgs_dark)
    ODs = []
    for i in 1:length(imgs_atoms)
        OD = (imgs_atoms[i] .- imgs_dark[i]) ./ (imgs_bkg[i] .- imgs_dark[i])
        OD[OD .< 0] .= NaN # Values where dark is brighter than laser/atoms are not taken into account
        OD .= -log10.(OD)
        push!(ODs, OD)
    end
    return ODs
end


In [ ]:
pathdir_date = "data/04_07_25/"
dir_names_datasets = readdir(pathdir_date)
dir_names_datasets = dir_names_datasets[dir_names_datasets .!= ".DS_Store"]
x_crop = [1100:1700;]
y_crop = [1700:2400;];

# Import and crop the images

In [ ]:
imgs_atoms_crop_datasets, imgs_bkg_crop_datasets, imgs_dark_crop_datasets = [], [], []
for dir_name_datasets in ProgressBar(dir_names_datasets)
    files_path = readdir(pathdir_date*dir_name_datasets)
    imgs_atoms, imgs_bkg, imgs_dark = load_imgs(files_path, pathdir_date*dir_name_datasets*"/")
    imgs_atoms, imgs_bkg, imgs_dark = crop_imgs(imgs_atoms, x_crop, y_crop), crop_imgs(imgs_bkg, x_crop, y_crop), crop_imgs(imgs_dark, x_crop, y_crop)
    push!(imgs_atoms_crop_datasets, imgs_atoms), push!(imgs_bkg_crop_datasets, imgs_bkg), push!(imgs_dark_crop_datasets, imgs_dark)
end

In [ ]:
@save "Imgs_Crop.jld2" imgs_atoms_crop_datasets imgs_bkg_crop_datasets imgs_dark_crop_datasets

In [ ]:
# @load "Imgs_Crop.jld2" imgs_atoms_crop_datasets imgs_bkg_crop_datasets imgs_dark_crop_datasets

# Compute the ODs

In [ ]:
ODs_datasets = []
for i in ProgressBar(1:length(imgs_atoms_crop_datasets))
    push!(ODs_datasets, compute_OD(imgs_atoms_crop_datasets[i], imgs_bkg_crop_datasets[i], imgs_dark_crop_datasets[i]))
end

In [ ]:
@save "ODs.jld2" ODs_datasets

In [5]:
@load "ODs.jld2" ODs_datasets

1-element Vector{Symbol}:
 :ODs_datasets

# Save the ODs images

In [6]:
if !isdir("imgs")
    mkdir("imgs")
end
if !isdir("imgs/neg_ODs")
    mkdir("imgs/neg_ODs")
end
if !isdir("imgs/neg_ODs/"*split(pathdir_date, "/")[2])
    mkdir("imgs/neg_ODs/"*split(pathdir_date, "/")[2])
end

In [7]:
close("all")
fig, axs = subplots()

for (i, ODs) in ProgressBar(enumerate(ODs_datasets))
    if !isdir("imgs/neg_ODs/"*split(pathdir_date, "/")[2]*"/"*dir_names_datasets[i])
        mkdir("imgs/neg_ODs/"*split(pathdir_date, "/")[2]*"/"*dir_names_datasets[i])
    end
    for (j, OD) in enumerate(ODs)
        img = axs.imshow(OD, cmap="bone", vmin=-0.2, vmax=0) #, aspect="auto"
        cb = colorbar(img)
        savefig("imgs/neg_ODs/"*split(pathdir_date, "/")[2]*"/"*dir_names_datasets[i]*"/$(j-1).png")
        cb.remove()
        axs.clear()
    end
end
close("all")

0.0%┣                                              ┫ 0/3 [00:12<00:-35, -12s/it]
33.3%┣██████████████▍                            ┫ 1/3 [00:43<Inf:Inf, InfGs/it]
66.7%┣███████████████████████████████▍               ┫ 2/3 [01:17<01:17, 77s/it]
100.0%┣██████████████████████████████████████████████┫ 3/3 [01:41<00:00, 50s/it]
100.0%┣██████████████████████████████████████████████┫ 3/3 [01:41<00:00, 50s/it]


In [ ]:
ODs_datasets